In [4]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01-12-2009 07:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01-12-2009 07:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01-12-2009 07:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01-12-2009 07:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01-12-2009 07:45,1.25,13085.0,United Kingdom


In [3]:
print("Before:", df.shape)
df = df.drop_duplicates()
print("After:", df.shape)

Before: (1048575, 8)
After: (1014425, 8)


In [4]:
df.shape
df.isnull().sum()

Invoice             0
StockCode           0
Description      4265
Quantity            0
InvoiceDate         0
Price               0
Customer ID    228826
Country             0
dtype: int64

In [5]:
df = df.dropna(subset=["Customer ID"])

In [6]:
#removing negative values
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

In [7]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], dayfirst=True)

In [8]:
df = df.drop_duplicates()

In [9]:
df["TotalPrice"] = df["Quantity"] * df["Price"]

<h5><b>Removing outliers using IQR method on TotalPrice

In [11]:
Q1 = df['TotalPrice'].quantile(0.25)
Q3 = df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Before:", df.shape)
df = df[(df['TotalPrice'] >= lower_bound) & (df['TotalPrice'] <= upper_bound)]
print("After:", df.shape)

Before: (767369, 9)
After: (704716, 9)


In [12]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 704716 entries, 4 to 1048574
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      704716 non-null  object        
 1   StockCode    704716 non-null  object        
 2   Description  704716 non-null  object        
 3   Quantity     704716 non-null  int64         
 4   InvoiceDate  704716 non-null  datetime64[ns]
 5   Price        704716 non-null  float64       
 6   Customer ID  704716 non-null  float64       
 7   Country      704716 non-null  object        
 8   TotalPrice   704716 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 53.8+ MB


,Quantity,InvoiceDate,Price,Customer ID,TotalPrice
count,704716.000000,704716,704716.000000,704716.000000,704716.000000
mean,7.741466,2010-12-29 20:44:07.359985920,2.811323,15345.495407,12.136344
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,2.000000,2010-07-01 12:13:00,1.250000,13995.000000,4.250000
50%,4.000000,2010-11-29 16:41:00,1.950000,15298.000000,10.200000
75%,12.000000,2011-07-24 11:25:00,3.750000,16813.000000,17.400000
max,900.000000,2011-12-04 13:15:00,41.090000,18287.000000,42.000000
std,10.097487,NaN,2.879778,1691.506433,9.054937


**Visualizing Outliers** (to avoid extreme values) : <br>
Lower bound (1%) <br>
Upper bound (99%)

In [13]:
df["Quantity"].quantile([0.01, 0.99])
df["Price"].quantile([0.01, 0.99])

0.01     0.29
0.99    12.75
Name: Price, dtype: float64

**Filtering Outliers** (using percentile-based filtering ): <br>
Reduces skewness and improves model performance.

In [14]:
#for Quantity
q_low = df["Quantity"].quantile(0.01)
q_high = df["Quantity"].quantile(0.99)

df = df[(df["Quantity"] >= q_low) & (df["Quantity"] <= q_high)]

In [15]:
#for price 
p_low = df["Price"].quantile(0.01)
p_high = df["Price"].quantile(0.99)

df = df[(df["Price"] >= p_low) & (df["Price"] <= p_high)]

FEATURE ENGINEERING <br>
Creating time-based features for analysis.

In [16]:
df["TotalPrice"] = df["Quantity"] * df["Price"]

In [17]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID,TotalPrice
count,689735.000000,689735,689735.000000,689735.000000,689735.000000
mean,7.274276,2010-12-29 10:16:22.127367936,2.707981,15349.779140,12.033967
min,1.000000,2009-12-01 07:45:00,0.290000,12346.000000,0.290000
25%,2.000000,2010-07-01 09:57:30,1.250000,13999.000000,4.250000
50%,4.000000,2010-11-29 14:21:00,1.950000,15311.000000,10.200000
75%,12.000000,2011-07-22 13:59:00,3.750000,16814.000000,17.000000
max,48.000000,2011-12-04 13:15:00,12.750000,18287.000000,42.000000
std,7.801213,NaN,2.433587,1690.407011,8.955534


In [18]:
df["Month"] = df["InvoiceDate"].dt.month
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek

In [19]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)